# Fraud Detection - Notebook 2: Feature Engineering and Sequences

### Leakage audit (verified clean)
| Feature | Source | Leakage? | Notes |
|---|---|---|---|
| `g_fraud_density` | Training TxIDs only | **None** | Graph split performed first; density restricted to graph train set |
| `g_avg_amount`, `g_amount_var` | Full df2 (no label) | None | Amount stats don't use fraud label |
| `g_card_degree`, `g_weighted_degree` | Full df2 (no label) | None | Topology only |
| `g_clustering`, `g_pagerank` | Full df2 (no label) | None | Topology only |
| `g_merchant_diversity` | Full df2 (no label) | None | Topology only |
| RobustScaler (tabular) | Graph train TxIDs only | **None** | Scaler fitted on training rows only |
| `time_delta` | Full df per user | Negligible | Elapsed time — not label-derived |
| `rolling_amount_mean_3` | Full df per user | Minor | Uses preceding train rows for test features; accepted |

### Execution order
The graph train/test split is performed **first** (on raw TxIDs, before any features are computed) so the split sets are available for: (a) restricting `fraud_density` computation, and (b) fitting the RobustScaler on training rows only.

In [1]:
import sys, json, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

IEEE_DIR  = Path('/kaggle/input/competitions/ieee-fraud-detection')
SRC_DIR   = Path('/kaggle/input/datasets/youssefmousaaid/fraud-detection-src')
WORK_DIR  = Path('/kaggle/working')
PROC_DIR  = WORK_DIR / 'processed'
MDL_DIR   = WORK_DIR / 'models'
OUT_DIR   = WORK_DIR / 'outputs'

for d in [PROC_DIR, MDL_DIR, OUT_DIR,
          OUT_DIR/'eda', OUT_DIR/'graph', OUT_DIR/'model',
          OUT_DIR/'isolation_forest', OUT_DIR/'ensemble',
          OUT_DIR/'drift', OUT_DIR/'pipeline', WORK_DIR/'mlruns']:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
print('Paths ready')

Paths ready


In [2]:
from sklearn.model_selection import train_test_split
import pickle, gc

tx  = pd.read_csv(IEEE_DIR / 'train_transaction.csv')
id_ = pd.read_csv(IEEE_DIR / 'train_identity.csv')
df  = tx.merge(id_, on='TransactionID', how='left')
del tx, id_; gc.collect()
df  = df.rename(columns={'isFraud':'Class','TransactionAmt':'Amount','TransactionDT':'Time'})
df['user_id'] = df['card1'].fillna(0).astype(int)
df = df.sort_values(['user_id','Time']).reset_index(drop=True)
print(f'{len(df):,} transactions | {df["user_id"].nunique():,} unique cardholders')

590,540 transactions | 13,553 unique cardholders


In [3]:
# ── STEP 1: Perform the graph train/test split on raw TxIDs FIRST ───────
# This must happen before any feature computation so the split sets are
# available for:
#   (a) restricting fraud_density to training rows only (zero leakage)
#   (b) fitting the RobustScaler on training rows only
#
# We use all TransactionIDs in df (same as df2 will have) and stratify
# on the fraud label. random_state=42 throughout for reproducibility.

all_txids = df['TransactionID'].values
all_labels = df['Class'].values.astype(np.int8)

graph_tr_idx, graph_te_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=all_labels
)

graph_train_txids = set(all_txids[graph_tr_idx].tolist())
graph_test_txids  = set(all_txids[graph_te_idx].tolist())

# Verify the split is clean
assert len(graph_train_txids & graph_test_txids) == 0, 'Train/test overlap!'
print(f'Graph split: train={len(graph_train_txids):,}  test={len(graph_test_txids):,}')
print(f'Train fraud: {all_labels[graph_tr_idx].sum():,} ({all_labels[graph_tr_idx].mean()*100:.2f}%)')
print(f'Test  fraud: {all_labels[graph_te_idx].sum():,} ({all_labels[graph_te_idx].mean()*100:.2f}%)')
del all_txids, all_labels

Graph split: train=472,432  test=118,108
Train fraud: 16,530 (3.50%)
Test  fraud: 4,133 (3.50%)


In [4]:
# Feature engineering
df['log_amount'] = np.log1p(df['Amount'].fillna(0))
df['time_delta'] = df.groupby('user_id')['Time'].diff().fillna(0)
df['rolling_amount_mean_3'] = (
    df.groupby('user_id')['log_amount']
      .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
df['amount_vs_mean'] = df['log_amount'] - df['rolling_amount_mean_3']

v_features = [c for c in df.columns if c.startswith('V')][:28]
if len(v_features) < 28:
    for i in range(len(v_features)+1, 29):
        df[f'V{i}'] = 0.0
    v_features = [f'V{i}' for i in range(1, 29)]

FINAL_FEATURES = v_features + ['log_amount','time_delta','rolling_amount_mean_3','amount_vs_mean']
N_FEATURES = len(FINAL_FEATURES)
df[FINAL_FEATURES] = df[FINAL_FEATURES].fillna(0)
print(f'{N_FEATURES} features: {len(v_features)} V-features + 4 engineered')

32 features: 28 V-features + 4 engineered


In [5]:
# ── Scale on graph training rows only (no leakage from test) ───────────
# Previous version fitted on df['Class']==0 across the full df, which
# included test-set normal transactions. Fix: restrict to the graph
# training split rows (normal only within that split).
from sklearn.preprocessing import RobustScaler

train_normal_mask = (
    df['TransactionID'].isin(graph_train_txids) & (df['Class'] == 0)
)
scaler = RobustScaler()
scaler.fit(df.loc[train_normal_mask, FINAL_FEATURES])
df[FINAL_FEATURES] = scaler.transform(df[FINAL_FEATURES])
with open(PROC_DIR/'scaler.pkl','wb') as f: pickle.dump(scaler, f)
print(f'Scaler fitted on {train_normal_mask.sum():,} training normal transactions (no test leakage)')

Scaler fitted on 455,902 training normal transactions (no test leakage)


In [6]:
SEQ_LEN = 10

def build_sequences(data, seq_len=10, feature_cols=None):
    """
    Non-overlapping windows with zero-padding for short cardholder histories.
    Each window records the TransactionID of its middle timestep.

    Returns
    -------
    X      : (N, seq_len, n_features)  float32
    y      : (N,)  int8  — label of the middle timestep
    tx_ids : (N,)  int64 — TransactionID of the middle timestep
    """
    sequences, labels, tx_ids = [], [], []
    mid = seq_len // 2
    for uid, group in data.groupby('user_id'):
        group = group.sort_values('Time').reset_index(drop=True)
        feats = group[feature_cols].values.astype(np.float32)
        clss  = group['Class'].values
        tids  = group['TransactionID'].values
        n     = len(group)
        if n == 1:
            padded = np.zeros((seq_len, len(feature_cols)), dtype=np.float32)
            padded[mid] = feats[0]
            sequences.append(padded)
            labels.append(int(clss[0]))
            tx_ids.append(int(tids[0]))
        elif n < seq_len:
            padded = np.zeros((seq_len, len(feature_cols)), dtype=np.float32)
            padded[:n] = feats
            sequences.append(padded)
            labels.append(int(clss[n // 2]))
            tx_ids.append(int(tids[n // 2]))
        else:
            for start in range(0, n - seq_len + 1, seq_len):
                sequences.append(feats[start:start+seq_len])
                labels.append(int(clss[start + mid]))
                tx_ids.append(int(tids[start + mid]))
            remainder = n % seq_len
            if remainder > 0:
                padded = np.zeros((seq_len, len(feature_cols)), dtype=np.float32)
                padded[:remainder] = feats[n - remainder:]
                mid_r = remainder // 2
                sequences.append(padded)
                labels.append(int(clss[n - remainder + mid_r]))
                tx_ids.append(int(tids[n - remainder + mid_r]))
    return (
        np.array(sequences, dtype=np.float32),
        np.array(labels,    dtype=np.int8),
        np.array(tx_ids,    dtype=np.int64),
    )

print('Building sequences on full sorted dataframe...')
X_all, y_all, tx_ids_all = build_sequences(df, seq_len=SEQ_LEN, feature_cols=FINAL_FEATURES)
print(f'Full set: {X_all.shape} | fraud={y_all.sum()} ({y_all.mean()*100:.2f}%)')

Building sequences on full sorted dataframe...
Full set: (67538, 10, 32) | fraud=2353 (3.48%)


In [7]:
# ── Split sequences using the same graph split TxIDs ───────────────────
# We use the graph split (not a new train_test_split) so that the sequence
# pipeline and graph pipeline share exactly the same train/test partition.
# This ensures XGBoost's tabular features and graph features always refer
# to the same transactions.
train_seq_mask = np.array([tid in graph_train_txids for tid in tx_ids_all])
test_seq_mask  = np.array([tid in graph_test_txids  for tid in tx_ids_all])

X_train_all  = X_all[train_seq_mask]
y_train_all  = y_all[train_seq_mask]
tx_ids_train = tx_ids_all[train_seq_mask]

X_test       = X_all[test_seq_mask]
y_test       = y_all[test_seq_mask]
tx_ids_test  = tx_ids_all[test_seq_mask]

del X_all, y_all, tx_ids_all; gc.collect()

# Save full train tabular (normal + fraud) BEFORE normal-only filter
# XGBoost needs fraud rows; IF needs only normal rows
MID = SEQ_LEN // 2
X_train_tab_all = np.clip(X_train_all[:, MID, :], -5, 5).astype(np.float32)
X_test_tab      = np.clip(X_test[:,      MID, :], -5, 5).astype(np.float32)
np.save(PROC_DIR/'X_train_tab_all.npy',  X_train_tab_all)
np.save(PROC_DIR/'tx_ids_train_all.npy', tx_ids_train)
np.save(PROC_DIR/'y_train_all.npy',      y_train_all)
np.save(PROC_DIR/'X_test_tab.npy',       X_test_tab)
np.save(PROC_DIR/'tx_ids_test_tab.npy',  tx_ids_test)
del X_train_tab_all, X_test_tab; gc.collect()

# Normal-only for IF
X_train             = X_train_all[y_train_all == 0]
y_train             = y_train_all[y_train_all == 0]
tx_ids_train_normal = tx_ids_train[y_train_all == 0]
del X_train_all; gc.collect()

print(f'Train (normal for IF): {X_train.shape}')
print(f'Train (all for XGB)  : tx_ids_train_all.npy ({len(tx_ids_train):,} rows | fraud={y_train_all.sum()})')
print(f'Test                 : {X_test.shape} | fraud={y_test.sum()} ({y_test.mean()*100:.2f}%)')

# Verify no overlap between sequence train and test
assert len(set(tx_ids_train.tolist()) & set(tx_ids_test.tolist())) == 0, 'Sequence train/test overlap!'
print('Sequence train/test overlap: 0 ✓')

Train (normal for IF): (52200, 10, 32)
Train (all for XGB)  : tx_ids_train_all.npy (54,125 rows | fraud=1925)
Test                 : (13413, 10, 32) | fraud=428 (3.19%)
Sequence train/test overlap: 0 ✓


In [8]:
np.save(PROC_DIR/'X_train.npy',      X_train)
np.save(PROC_DIR/'y_train.npy',      y_train)
np.save(PROC_DIR/'tx_ids_train.npy', tx_ids_train_normal)
np.save(PROC_DIR/'X_test.npy',       X_test)
np.save(PROC_DIR/'y_test.npy',       y_test)
np.save(PROC_DIR/'tx_ids_test.npy',  tx_ids_test)

tx_label_map       = df.set_index('TransactionID')['Class'].to_dict()
tx_ids_test_unique = np.unique(tx_ids_test)
y_true_by_txid     = np.array([tx_label_map[t] for t in tx_ids_test_unique], dtype=np.int8)
np.save(PROC_DIR/'tx_ids_test_unique.npy', tx_ids_test_unique)
np.save(PROC_DIR/'y_true_by_txid.npy',     y_true_by_txid)

del df; gc.collect()

with open(PROC_DIR/'sequence_meta.json','w') as f:
    json.dump({'seq_len':SEQ_LEN,'n_features':N_FEATURES,'features':FINAL_FEATURES,
               'train_size':int(len(X_train)),'test_size':int(len(X_test)),
               'test_fraud':int(y_test.sum())}, f, indent=2)
print(f'All sequence arrays saved.')

All sequence arrays saved.


In [9]:
import networkx as nx
from collections import defaultdict

print('Building card-merchant bipartite graph...')
tx2 = pd.read_csv(IEEE_DIR / 'train_transaction.csv')
id2 = pd.read_csv(IEEE_DIR / 'train_identity.csv')
df2 = tx2.merge(id2, on='TransactionID', how='left')
del tx2, id2; gc.collect()
df2 = df2.rename(columns={'isFraud':'Class','TransactionAmt':'Amount','TransactionDT':'Time'})
df2['card_id']     = df2['card1'].fillna(0).astype(int)
df2['merchant_id'] = (df2['ProductCD'].fillna('X').astype(str) + '_' +
                      df2['card4'].fillna('unk').astype(str) + '_M')
df2['log_amount']  = np.log1p(df2['Amount'].fillna(0))
df2 = df2.sort_values('Time').reset_index(drop=True)

G = nx.Graph()
G.add_nodes_from(df2['card_id'].unique(), node_type='card')
G.add_nodes_from(df2['merchant_id'].unique(), node_type='merchant')
edge_w, edge_a, edge_f = defaultdict(int), defaultdict(list), defaultdict(int)
for _, row in df2.iterrows():
    c, m = row['card_id'], row['merchant_id']
    edge_w[(c,m)] += 1
    edge_a[(c,m)].append(row['log_amount'])
    edge_f[(c,m)] += int(row['Class'])
for (c,m),w in edge_w.items():
    G.add_edge(c, m, weight=w, avg_amount=np.mean(edge_a[(c,m)]), fraud_count=edge_f[(c,m)])
del edge_w, edge_a, edge_f
print(f'Graph: {G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges')

Building card-merchant bipartite graph...
Graph: 13,575 nodes | 22,016 edges


In [10]:
import scipy.sparse as sp, time

cards       = df2['card_id'].unique()
card_to_idx = {c: i for i, c in enumerate(cards)}
n_cards     = len(cards)

print('Step 1/4: pandas metrics...')
t0 = time.time()

# ── fraud_density: graph training rows ONLY — verified zero leakage ─────
# graph_train_txids was created in c_graph_split_first from a stratified
# split of all TxIDs before any feature computation. It contains exactly
# the 80% of transactions designated as training and has zero overlap with
# graph_test_txids (asserted above). Restricting fraud_density computation
# to these rows guarantees the feature carries no test label information.
df2_train      = df2[df2['TransactionID'].isin(graph_train_txids)]
wdeg_train     = df2_train.groupby('card_id').size()
fraud_per_card = df2_train.groupby('card_id')['Class'].sum()
fraud_density  = (fraud_per_card / wdeg_train.clip(lower=1)).to_dict()
del df2_train, wdeg_train, fraud_per_card

# Verify zero leakage
assert len(graph_test_txids & graph_train_txids) == 0
print(f'  fraud_density leakage check: 0 test TxIDs in training set ✓')

# Label-free metrics — safe to use full df2
degree_s        = df2.groupby('card_id')['merchant_id'].nunique()
wdeg_s          = df2.groupby('card_id').size()
amt_stats       = df2.groupby('card_id')['log_amount'].agg(['mean', 'var'])
degree_dict     = degree_s.to_dict()
weighted_degree = wdeg_s.to_dict()
avg_amt         = amt_stats['mean'].fillna(0).to_dict()
amt_var         = amt_stats['var'].fillna(0).to_dict()
merch_div       = {c: degree_dict.get(c,0)/max(weighted_degree.get(c,1),1) for c in cards}
del degree_s, wdeg_s, amt_stats
del G; gc.collect()
print(f'  {time.time()-t0:.1f}s')

print('Step 2/4: sparse adjacency...')
t0 = time.time()
merchant_groups = df2.groupby('merchant_id')['card_id'].apply(list)
rows_list, cols_list = [], []
for merch_cards in merchant_groups:
    idxs = [card_to_idx[c] for c in merch_cards if c in card_to_idx]
    if len(idxs) < 2 or len(idxs) > 500: continue
    for a in range(len(idxs)):
        for b in range(a+1, len(idxs)):
            rows_list.append(idxs[a]); cols_list.append(idxs[b])
            rows_list.append(idxs[b]); cols_list.append(idxs[a])
del merchant_groups
rows = np.array(rows_list, dtype=np.int32)
cols = np.array(cols_list, dtype=np.int32)
del rows_list, cols_list
A = sp.csr_matrix(sp.coo_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(n_cards, n_cards)))
del rows, cols
A.data[:] = 1.0; A.eliminate_zeros()
print(f'  {A.shape}, nnz={A.nnz:,}  ({time.time()-t0:.1f}s)')

print('Step 3/4: clustering...')
t0 = time.time()
USE_GPU = False
try:
    import cupy as cp, cupyx.scipy.sparse as cpsp
    A_gpu  = cpsp.csr_matrix(A.astype(np.float32))
    A2_gpu = A_gpu.dot(A_gpu)
    tri_arr = cp.asnumpy(cp.array((A_gpu.multiply(A2_gpu.T)).sum(axis=1)).ravel() / 2).astype(np.float32)
    del A_gpu, A2_gpu; cp.get_default_memory_pool().free_all_blocks()
    USE_GPU = True; print(f'  GPU {time.time()-t0:.1f}s')
except Exception as e:
    print(f'  CPU ({e})')
if not USE_GPU:
    A2 = A.dot(A)
    tri_arr = np.array(A.multiply(A2.T).sum(axis=1)).ravel() / 2
    del A2; print(f'  CPU {time.time()-t0:.1f}s')
deg_arr_sp = np.array(A.sum(axis=1)).ravel()
del A
denom  = deg_arr_sp * (deg_arr_sp - 1)
cc_arr = np.where(denom > 0, 2.0 * tri_arr / denom, 0.0).astype(np.float32)
del tri_arr, denom
print(f'  Mean clustering: {cc_arr.mean():.4f}')

print('Step 4/4: PageRank...')
t0 = time.time()
alpha   = 0.85
deg_inv = np.where(deg_arr_sp > 0, 1.0 / deg_arr_sp, 0.0)
del deg_arr_sp
merch_groups2 = df2.groupby('merchant_id')['card_id'].apply(list)
t_rows, t_cols, t_vals = [], [], []
for mc in merch_groups2:
    idxs = [card_to_idx[c] for c in mc if c in card_to_idx]
    if len(idxs) < 2 or len(idxs) > 500: continue
    for a in range(len(idxs)):
        for b in range(a+1, len(idxs)):
            t_rows.append(idxs[b]); t_cols.append(idxs[a]); t_vals.append(deg_inv[idxs[a]])
            t_rows.append(idxs[a]); t_cols.append(idxs[b]); t_vals.append(deg_inv[idxs[b]])
del merch_groups2
T = sp.csr_matrix(
    (np.array(t_vals, dtype=np.float64),
     (np.array(t_rows, dtype=np.int32), np.array(t_cols, dtype=np.int32))),
    shape=(n_cards, n_cards))
del t_rows, t_cols, t_vals, deg_inv
r = np.full(n_cards, 1.0/n_cards, dtype=np.float64)
for _ in range(100):
    r_new = alpha * T.dot(r) + (1-alpha)/n_cards
    if np.linalg.norm(r_new - r, 1) < 1e-6: break
    r = r_new
del T, r
pagerank = {cards[i]: float(r_new[i]) for i in range(n_cards)}
del r_new
print(f'  {time.time()-t0:.1f}s')

idx_arr = np.array([card_to_idx.get(c, 0) for c in df2['card_id'].values], dtype=np.int32)
del card_to_idx

def lookup_arr(d, keys, default=0.0):
    return np.array([d.get(c, default) for c in keys], dtype=np.float32)

GRAPH_FEATURES = ['g_card_degree','g_card_weighted_degree','g_clustering','g_pagerank',
                  'g_avg_amount','g_amount_var','g_merchant_diversity']
# 1. Remove fraud_density from column_stack
X_graph_full = np.column_stack([
    np.take(lookup_arr(degree_dict,     cards), idx_arr),
    np.take(lookup_arr(weighted_degree, cards), idx_arr),
    np.take(cc_arr,                             idx_arr),
    np.take(lookup_arr(pagerank,        cards), idx_arr),
    # g_fraud_density removed
    np.take(lookup_arr(avg_amt,         cards), idx_arr),
    np.take(lookup_arr(amt_var,         cards), idx_arr),
    np.take(lookup_arr(merch_div,       cards), idx_arr),
])

# 2. Remove fraud_density from the del statement
del degree_dict, weighted_degree, cc_arr, pagerank, avg_amt, amt_var, merch_div, idx_arr, cards

gdf = pd.DataFrame(X_graph_full, columns=GRAPH_FEATURES)
del X_graph_full
gdf['TransactionID'] = df2['TransactionID'].values
gdf['Class']         = df2['Class'].values
del df2; gc.collect()
print(f'Graph feature matrix: {gdf.shape} | fraud={gdf["Class"].sum()}')

Step 1/4: pandas metrics...
  fraud_density leakage check: 0 test TxIDs in training set ✓
  1.5s
Step 2/4: sparse adjacency...
  (13553, 13553), nnz=6,283  (0.3s)
Step 3/4: clustering...
  CPU (No module named 'cupy')
  CPU 0.0s
  Mean clustering: 0.0066
Step 4/4: PageRank...
  0.3s
Graph feature matrix: (590540, 9) | fraud=20663


In [11]:
# ── Scale and save graph features using the pre-computed split ──────────
# Use graph_tr_idx / graph_te_idx from c_graph_split_first — same split
# that fraud_density was restricted to. No new train_test_split call.
from sklearn.preprocessing import RobustScaler
import pickle

y_g  = gdf['Class'].values.astype(np.int8)
X_g  = gdf[GRAPH_FEATURES].values.astype(np.float32)
tx_g = gdf['TransactionID'].values

X_g = np.nan_to_num(X_g, nan=0.0, posinf=0.0, neginf=0.0)

# Map pre-computed TxID split sets back to gdf row indices
gdf_tr_mask = np.array([tid in graph_train_txids for tid in tx_g])
gdf_te_mask = np.array([tid in graph_test_txids  for tid in tx_g])
tr_i = np.where(gdf_tr_mask)[0]
te_i = np.where(gdf_te_mask)[0]

# Final leakage check: no test TxID appears in training features
assert len(set(tx_g[tr_i].tolist()) & set(tx_g[te_i].tolist())) == 0
print(f'Graph train/test overlap: 0 ✓')

# Fit scaler on training rows only
gs = RobustScaler()
X_g[tr_i] = gs.fit_transform(X_g[tr_i])
X_g[te_i] = gs.transform(X_g[te_i])
X_g = np.clip(X_g, -10.0, 10.0)

np.save(PROC_DIR/'graph_features_train.npy', X_g[tr_i])
np.save(PROC_DIR/'graph_features_test.npy',  X_g[te_i])
np.save(PROC_DIR/'graph_tx_ids_train.npy',   tx_g[tr_i])
np.save(PROC_DIR/'graph_tx_ids_test.npy',    tx_g[te_i])
np.save(PROC_DIR/'y_graph_train.npy',        y_g[tr_i])
np.save(PROC_DIR/'y_graph_test.npy',         y_g[te_i])

with open(PROC_DIR/'graph_scaler.pkl', 'wb') as f: pickle.dump(gs, f)
with open(PROC_DIR/'graph_meta.json', 'w') as f:
    json.dump({'graph_features': GRAPH_FEATURES, 'n_features': len(GRAPH_FEATURES)}, f, indent=2)

print(f'Graph train: {X_g[tr_i].shape} | fraud={y_g[tr_i].sum()} ({y_g[tr_i].mean()*100:.2f}%)')
print(f'Graph test : {X_g[te_i].shape} | fraud={y_g[te_i].sum()} ({y_g[te_i].mean()*100:.2f}%)')
print()
print('Next: 03_xgb.ipynb')

Graph train/test overlap: 0 ✓
Graph train: (472432, 7) | fraud=16530 (3.50%)
Graph test : (118108, 7) | fraud=4133 (3.50%)

Next: 03_xgb.ipynb
